# Transaction Foundation Model on Ray — Part 2: Load & explore the data w/ Ray Data

<div align="left">
  <a target="_blank" href="https://console.anyscale.com/template-preview/fintech_transaction_fm"><img src="https://img.shields.io/badge/🚀 Run_on-Anyscale-9hf"></a>&nbsp;
  <a href="https://github.com/anyscale/templates/tree/main/templates/fintech_transaction_fm" role="button"><img src="https://img.shields.io/static/v1?label=&message=View%20On%20GitHub&color=586069&logo=github&labelColor=2f363d"></a>&nbsp;
</div>

**⏱️ Time to complete**: ~15 min (most of it the one-time dataset download + normalize)


---

In Part 1, we set up the cluster. Here in Part 2, we load the benchmark dataset

In [1]:
import sys, os, json

DEMO_ROOT = os.path.abspath(os.getcwd())
if DEMO_ROOT not in sys.path:
    sys.path.insert(0, DEMO_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import ray
ray.init(ignore_reinit_error=True, runtime_env={"working_dir": DEMO_ROOT})

2026-06-25 15:56:07,363	INFO worker.py:1814 -- Connecting to existing Ray cluster at address: 10.0.113.42:6379...
2026-06-25 15:56:07,394	INFO worker.py:2003 -- Connected to Ray cluster. View the dashboard at https://session-nridlis6nyy2hi9qv4il5lcj6q.i.anyscaleuserdata.com 
2026-06-25 15:56:07,410	INFO packaging.py:691 -- Creating a file package for local module '/home/ray/default_cld_g54aiirwj1s8t9ktgzikqur41k/templates/templates/fintech_transaction_fm'.
2026-06-25 15:56:07,422	INFO packaging.py:463 -- Pushing file package 'gcs://_ray_pkg_6def4dfcab711af1.zip' (0.40MiB) to Ray cluster...
2026-06-25 15:56:07,424	INFO packaging.py:476 -- Successfully pushed file package 'gcs://_ray_pkg_6def4dfcab711af1.zip'.
/home/ray/anaconda3/lib/python3.12/site-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error mes

Python version:,3.12.13
Ray version:,2.55.1
Dashboard:,http://session-nridlis6nyy2hi9qv4il5lcj6q.i.anyscaleuserdata.com


## IBM TabFormer Dataset

We use the **IBM TabFormer** dataset (Padhi et al., ICASSP 2021) — the de-facto public benchmark for transaction foundation models. It has 24.4M card transactions, ~6.1k cards, 1991–2020, ~0.12% transaction-level fraud. NVIDIA's transaction-FM blueprint and FATA-Trans both evaluate on it, so our downstream numbers are comparable.


We load the 24.4M rows as a Ray Data Pipeline that streams across the cluster. Something like Pandas might OOM on this relatively small dataset, but Ray Data, allows us to load datasets of any scale by distributing it through a scalable cluster.

General approach:
1. **Read: `read_csv`** — read the raw file in parallel
2. **Normalize: `map_batches(normalize_batch)`** — parse each batch into our canonical schema (things like - `"$57.20"` → `57.20`, MCC → category, modal home state — lives in `src/tabformer.py`; it's dataset-specific and a bit in the weeds).
3. **Aggregate: `groupby("card_id").map_groups(card_statics)`** — a distributed aggregation for the per-card fields. These "static" fields are broadcast back into each transation row.
4. **Write out: `write_parquet`** — a streamed write to sharded Parquet.

Some dataset-specific callbacks are imported `src/tabformer.py` and we compose the pipeline here. See `tabformer.prepare_tabformer()` for the identical pipeline wrapped for headless (scripts/job) runs.

### A note on `SCALE` config

`SCALE` is one knob that defines parameters for the run. In this notebook, the main parameter we use is how many cards to sample; in later notebooks, the sequence length, model size, and number of GPU workers. The three presets (`mini` / `small` / `full`) live in `configs/`. We use **`mini`**: tiny and CPU-only, so this notebook runs in minutes. It's the *same code, change config* idea — flip the variable to go from a laptop-sized run to a full multi-GPU one. Outputs are written under a per-scale path (`.../raw/mini/…`) so different scales don't overwrite each other.

### Build the Canonical Dataset, including a 80/10/10 Temporal Split

In this section, we read the data and normalize it with a map_batches call. 

Then we create the 80/10/10 temporal cutoffs. This approach means that the model is trained off of the earliest 80% of the transactions, the next 10% is the validation set, the final (most recent) 10% is the test set.

In [2]:
from src.paths import artifact_paths, get_demo_base_dir, write_splits_meta
from src.scale_config import load_scale
from src.tabformer import (
    ensure_download, normalize_batch, card_statics,
    sample_cards, attach_statics, tabformer_csv_convert_options,
)

SCALE = "mini"            # which configs/<SCALE>.yaml preset to run; mini = tiny, CPU-only
cfg = load_scale(SCALE)    # the configs/mini.yaml settings (num_cards here; model/training later)
paths = artifact_paths(get_demo_base_dir(), SCALE)   # outputs namespaced per scale: .../<stage>/mini/

# Build the canonical dataset and save it; re-runs reuse the cached Parquet shards.
if not (os.path.exists(paths["raw"]) and os.path.exists(paths["splits"])):
    csv_path = ensure_download(paths["source"])   # one-time ~266MB download + extract

    # Read 24.4M rows and normalize to our schema — streamed across the cluster.
    ds = ray.data.read_csv(
        csv_path, convert_options=tabformer_csv_convert_options(),
    ).map_batches(normalize_batch, batch_format="pandas")
    ds = sample_cards(ds, cfg["data"]["num_cards"]).materialize()   # mini -> 2,000 cards; cache the read (reused below)

    # Per-card STATIC fields via a distributed groupby, broadcast onto every txn.
    statics = ds.groupby("card_id").map_groups(card_statics, batch_format="pandas").to_pandas()
    ds = attach_statics(ds, statics).materialize()          # cache before the two consumers below

    ds.write_parquet(paths["raw"])                 # streamed write -> Parquet shards

    # Build the Temporal 80/10/10 cutoffs that later stage reuses. This approach means that the model is trained off of 
    # the earliest 80% of the transactions, the next 10% is the validation set, the final (most recent) 10% is the test set.
    #
    # Note we only need 2 tiny columns for the splits, not the whole table, so there's no OOM risk 
    # allowing us to do the pragmatic thing of splitting with a to_pandas call which brings the data to the driver 
    # and avoiding a full dataset sort
    slim = ds.select_columns(["timestamp", "is_fraud"]).to_pandas()
    write_splits_meta(paths["splits"], slim["timestamp"].to_numpy(),
                      slim["is_fraud"].to_numpy(), source="tabformer", n_cards=len(statics))

In [3]:
# Load the normalized transactions back, sorted into per-card time order.
df = pd.read_parquet(paths["raw"]).sort_values(["card_id", "timestamp"])
with open(paths["splits"]) as f:
    splits = json.load(f)

print(f"{len(df):,} transactions  /  {df['card_id'].nunique():,} cards  "
      f"/  {df['is_fraud'].mean()*100:.3f}% fraud")
df.head(5)

7,761,979 transactions  /  2,000 cards  /  0.125% fraud


,card_id,timestamp,amount,merchant_id,merchant_category,mcc,hour,day_of_week,is_fraud,issuer,bin_region,card_type,home_state
5124627,101,2011-02-02 13:20:00,47.64,6698459923198770712,utilities,4814,13,2,0,UNKNOWN,UNKNOWN,chip,NY
5124628,101,2011-02-02 22:06:00,120.00,-4282466774399734331,utilities,4829,22,2,0,UNKNOWN,UNKNOWN,chip,NY
5124629,101,2011-02-03 07:29:00,18.86,1799189980464955940,grocery,5499,7,3,0,UNKNOWN,UNKNOWN,chip,NY
5124630,101,2011-02-05 06:55:00,109.28,-5904116920141006298,retail,5719,6,5,0,UNKNOWN,UNKNOWN,chip,NY
5124631,101,2011-02-06 19:06:00,74.05,4318859768586538780,restaurant,5812,19,6,0,UNKNOWN,UNKNOWN,chip,NY


## The static / dynamic split — visible in the schema

Each transaction has two kinds of field:

- **Static** (card-level): `issuer`, `bin_region`, `card_type`, `home_state` — they describe the *card* and never change within its sequence.
- **Dynamic** (per-transaction): `amount`, `merchant_id`, `merchant_category`, `mcc`, `hour`, `day_of_week` — they change every event.

The tokenizer (Part 3) embeds the static fields **once** and broadcasts them to every position, while each dynamic field gets its own embedding table and occupies **one** position per transaction.

### Modeling Rationale

The static/dynamic split is a key characteristic of Transaction Foundation Models that does not exist in LLMs. 

A transaction foundation model treats one card's transaction history as a sequence, the same way a language model treats a sentence as a sequence of words. It's learning the grammar of how this card behaves: given everything it's done so far, what's a plausible next transaction, and does this one fit the pattern? To do that it needs two different kinds of information, and they play different structural roles.

The dynamic fields are the sequence. amount, merchant_id, merchant_category, mcc, hour, day_of_week which change at every event. The dynamic fields make up the behavior the model is trying to learn the rhythm of. They're the words. Order matters, values matter, and the model's attention runs over them.

The static fields are not part of the sequence. The card issuer, bin_region, card_type, etc describe the card itself, and do not change over multiple transactions. They carry zero sequential information: there's nothing to learn over time about card=visa. So instead of letting them take up room in the timeline, the tokenizer embeds them once per card.

You /could/ treat these static fields like the dynamic fields, but it would cost a significant amount of precious sequence length. Since transformers are quadratic in sequence length, it's quite costly.

In [4]:
STATIC_COLS  = ["issuer", "bin_region", "card_type", "home_state"]
DYNAMIC_COLS = ["amount", "merchant_id", "merchant_category", "mcc", "hour", "day_of_week"]

# Max distinct values per card: statics collapse to 1, dynamics vary.
per_card_nunique = df.groupby("card_id")[STATIC_COLS + DYNAMIC_COLS].nunique().max()

print("kind      field                max distinct values per card")
print("-" * 56)
for c in STATIC_COLS:
    print(f"static    {c:<18}   {per_card_nunique[c]:>10,}")
for c in DYNAMIC_COLS:
    print(f"dynamic   {c:<18}   {per_card_nunique[c]:>10,}")

(autoscaler +6s) Tip: use `ray status` to view detailed cluster status. To disable these messages, set RAY_SCHEDULER_EVENTS=0.
kind      field                max distinct values per card
--------------------------------------------------------
static    issuer                        1
static    bin_region                    1
static    card_type                     1
static    home_state                    1
dynamic   amount                   11,887
dynamic   merchant_id                 726
dynamic   merchant_category             8
dynamic   mcc                          95
dynamic   hour                         24
dynamic   day_of_week                   7


Note `issuer` and `bin_region` are constant (`UNKNOWN`) because TabFormer doesn't carry them — the loader derives `card_type` and `home_state` per card and leaves the rest as a documented `UNKNOWN` bucket. On richer real data these static fields carry real signal, and the split pays off more.

## One card is one sequence

The model learns a card's transactions as a time-ordered sequence. Here is what one looks like, these are the dynamic fields the model will learn to predict (masked-feature modeling, Part 4).

In [5]:
card = df["card_id"].iloc[0]
seq = df[df["card_id"] == card]
print(f"card {card}: {len(seq):,} transactions from "
      f"{seq['timestamp'].min().date()} to {seq['timestamp'].max().date()}")
seq[["timestamp", "amount", "merchant_category", "mcc",
     "hour", "day_of_week", "is_fraud"]].head(10)

card 101: 1,830 transactions from 2011-02-02 to 2020-02-25


,timestamp,amount,merchant_category,mcc,hour,day_of_week,is_fraud
5124627,2011-02-02 13:20:00,47.64,utilities,4814,13,2,0
5124628,2011-02-02 22:06:00,120.00,utilities,4829,22,2,0
5124629,2011-02-03 07:29:00,18.86,grocery,5499,7,3,0
5124630,2011-02-05 06:55:00,109.28,retail,5719,6,5,0
5124631,2011-02-06 19:06:00,74.05,restaurant,5812,19,6,0
5124632,2011-02-10 12:33:00,24.63,utilities,4900,12,3,0
5124633,2011-02-13 19:03:00,67.21,utilities,4900,19,6,0
5124634,2011-02-14 04:56:00,139.57,utilities,4900,4,0,0
5124635,2011-02-17 12:51:00,26.60,other,7349,12,3,0
5124636,2011-02-19 13:31:00,102.39,other,7538,13,5,0


## Dataset shape and the temporal split

A final review of the data stats and temporal splits. 

In [6]:
seq_lens = df.groupby("card_id").size()
print(f"transactions ............ {len(df):,}")
print(f"cards ................... {df['card_id'].nunique():,}")
print(f"txns per card ........... median {int(seq_lens.median()):,}  "
      f"(min {seq_lens.min():,}, max {seq_lens.max():,})")
print(f"date range .............. {df['timestamp'].min().date()} -> {df['timestamp'].max().date()}")
print(f"overall fraud rate ...... {df['is_fraud'].mean()*100:.3f}%  "
      f"({int(df['is_fraud'].sum()):,} fraudulent txns)")
print()
print("temporal split (from splits.json — the same cutoffs every stage uses):")
print(f"  train  <  {splits['train_end']}")
print(f"  val    <  {splits['val_end']}")
print(f"  test   >= {splits['val_end']}")

transactions ............ 7,761,979
cards ................... 2,000
txns per card ........... median 2,536  (min 2, max 49,261)
date range .............. 1991-07-01 -> 2020-02-28
overall fraud rate ...... 0.125%  (9,685 fraudulent txns)

temporal split (from splits.json — the same cutoffs every stage uses):
  train  <  2017-06-08T15:42:24
  val    <  2018-10-29T03:22:12
  test   >= 2018-10-29T03:22:12


## Inspecting Data Distributions

Before we build out a tokenizer or a model, we inspect the data with a few plots. Each one guides an upcoming design decision.

1. **Transactions per card.** Cards vary enormously in activity — from just a handful of transactions to nearly 50,000, with a typical (median) card around 2,500. That huge spread is what the *sequence length* has to handle: a window long enough to capture real behavioral context for active cards, but not so long that the many short-history cards (roughly a fifth have fewer than 100 transactions) become mostly padding.
2. **Transaction amount.** Amounts span several orders of magnitude — from under a dollar to a few thousand — but cluster in the tens of dollars (median ~\$33), with a right-skewed tail of larger purchases. Because amount is multiplicative rather than additive (the step from \$10 to \$100 matters like \$100 to \$1,000), a raw scalar would spend almost all its resolution on the crowded low end. So we *bucket amounts on a log scale* and treat each bucket as its own categorical token.
3. **Volume over time.** Transaction volume grows steeply through the 2000s and then levels off, so an 80/10/10 split by transaction count puts the validation and test cutoffs in the most recent stretch of calendar time. That's what we want: splitting by time trains the model on the past and evaluates it on the future, so it never gets to peek ahead.
4. **Time between transactions.** The gaps are bursty — often only a few hours apart (median ~9 hours, a fifth of them within the hour), but stretching to days or, occasionally, far longer. So a position's ordinal slot ("the 5th transaction") throws away how much real time actually passed; we also embed *how long it's been since the previous transaction*, on a log scale, to make position time-aware.

In [ ]:
sns.set_theme(style="white", context="notebook")
from matplotlib.ticker import FuncFormatter

# Human-readable axis numbers: 600000 -> "600k", 2_000_000 -> "2M".
def _human(x, _):
    x = float(x)
    if abs(x) >= 1_000_000: return f"{x / 1e6:g}M"
    if abs(x) >= 1_000:     return f"{x / 1e3:g}k"
    return f"{x:g}"
human = FuncFormatter(_human)

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# (1) transactions per card -> motivates the sequence window length (seq_len).
# Log y so the long tail of high-activity cards stays visible under the tall first bar.
ax = axes[0, 0]
sns.histplot(seq_lens, bins=40, color="#4C78A8", ax=ax)
ax.set_yscale("log")
ax.set_title("Transactions per card (sequence length)")
ax.set_xlabel("transactions"); ax.set_ylabel("cards (log scale)")
ax.xaxis.set_major_formatter(human); ax.yaxis.set_major_formatter(human)

# (2) amount spans orders of magnitude -> we log-bucket it in the tokenizer.
ax = axes[0, 1]
log_amt = np.log10(np.clip(np.abs(df["amount"].to_numpy()), 0.1, None))
sns.histplot(log_amt, bins=60, color="#F58518", ax=ax)
ax.set_title("Transaction amount (log10 $) — spans orders of magnitude")
ax.set_xlabel("log10(amount)"); ax.set_ylabel("transactions")
ax.yaxis.set_major_formatter(human)

# (3) volume over time with the temporal 80/10/10 split boundaries.
ax = axes[1, 0]
monthly = df.groupby(df["timestamp"].dt.to_period("M")).size()
sns.lineplot(x=monthly.index.to_timestamp(), y=monthly.values, color="#54A24B", ax=ax)
for cut, lbl in [(splits["train_end"], "train|val"), (splits["val_end"], "val|test")]:
    ax.axvline(pd.Timestamp(cut), color="crimson", ls="--", lw=1)
    ax.text(pd.Timestamp(cut), ax.get_ylim()[1] * 0.92, lbl, rotation=90,
            va="top", ha="right", color="crimson", fontsize=8)
ax.set_title("Transaction volume over time + temporal split")
ax.set_xlabel("month"); ax.set_ylabel("transactions")
ax.yaxis.set_major_formatter(human)

# (4) inter-transaction gaps -> motivates time-aware position embeddings.
ax = axes[1, 1]
gaps = df.groupby("card_id")["timestamp"].diff().dt.total_seconds() / 3600.0
gaps = gaps.dropna()
gaps = gaps[gaps > 0]
sns.histplot(np.log10(gaps), bins=60, color="#B279A2", ax=ax)
ax.set_title("Hours between consecutive txns (log10) — bursty")
ax.set_xlabel("log10(hours since previous txn)"); ax.set_ylabel("transactions")
ax.yaxis.set_major_formatter(human)

sns.despine(fig=fig)
plt.tight_layout()
plt.show()

## Evaluations & how we measure performance

With only about 1 transaction in 800 fraudulent, 'accuracy' is meaningless — a model that flags everything as valid will score 99.9% accuracy and catches no fraud. We use PR-AUC (average precision) instead, which rewards ranking the rare fraud above everything else.

That same rarity shapes the eval set: we keep every fraud but drop most normal transactions, since scoring millions of obvious non-fraud just wastes compute. That leaves the set far more fraud-heavy than reality, so we apply 'importance weighting', which normalizes the evaluation metrics at the true ~0.12% rate.

In [24]:
fraud_rate = df["is_fraud"].mean()
print(f"normal transactions : {(1 - fraud_rate) * 100:.3f}%")
print(f"fraud transactions  : {fraud_rate * 100:.3f}%")
print(f"imbalance ratio     : ~1 fraud per {int(round(1 / fraud_rate)):,} transactions")

normal transactions : 99.875%
fraud transactions  : 0.125%
imbalance ratio     : ~1 fraud per 801 transactions


## Takeaways

**Ray Data**
- Loading the data is itself a distributed job, allowing parallel streaming of rows across the auto-scaling cluster, allowing fast loading of datasets of unlimited size.
- The same pipeline runs unchanged at various scales from laptop and fast-iteration friendly 'mini' scale to the `full` size (multi-GPU, every card); only the config changes.

**The data**
- One card = one **time-ordered sequence**; fields split cleanly into **static** (card-level, constant) and **dynamic** (per-transaction).
- The temporal split is computed once and written to `splits.json`, then reused by every later stage, so there's no leakage and runs stay comparable.
- The distributions drive the modeling choices: amounts span orders of magnitude (log-bucket them), inter-transaction gaps are bursty (time-aware positions), and card histories vary enormously (windowed sequences).
- Fraud is rare (~1 in 800), so we measure with PR-AUC and reweight the eval set back to the true rate.

---

## Next

**Part 3 — Tokenize with Ray Data**: turn these raw sequences into padded, per-field token windows using the static/dynamic split, as a stateless, embarrassingly parallel `map_groups` over the cluster.